In [38]:
using LowLevelFEM

In [39]:
openGeometry("furat.geo")

Info    : Clearing all models and views...
Info    : Done clearing all models and views
Info    : Reading 'furat.geo'...
Info    : Meshing 1D...                                                                                                                        
Info    : [  0%] Meshing curve 5 (Ellipse)
Info    : [ 30%] Meshing curve 6 (Line)
Info    : [ 50%] Meshing curve 7 (Line)
Info    : [ 70%] Meshing curve 8 (Line)
Info    : [ 90%] Meshing curve 9 (Line)
Info    : Done meshing 1D (Wall 0.00218577s, CPU 0.001515s)
Info    : Meshing 2D...
Info    : Meshing surface 1 (Plane, Frontal-Delaunay)
Info    : Blossom: 22788 internal 756 closed
Info    : Blossom recombination completed (Wall 1.0459s, CPU 1.03052s): 7553 quads, 0 triangles, 0 invalid quads, 0 quads with Q < 0.1, avg Q = 0.805118, min Q = 0.312842
Info    : Done meshing 2D (Wall 1.66425s, CPU 1.64541s)
Info    : Meshing order 4 (curvilinear on)...
Info    : [  0%] Meshing curve 5 order 4
Info    : [ 20%] Meshing curve 6 or

In [40]:
openPreProcessor()

-------------------------------------------------------
Version       : 4.15.0-git
License       : GNU General Public License
Build OS      : Linux64-sdk
Build date    : 19700101
Build host    : amdci7.julia.csail.mit.edu
Build options : 64Bit ALGLIB[contrib] ANN[contrib] Bamg Blossom Cairo DIntegration Dlopen DomHex Eigen[contrib] Fltk GMP Gmm[contrib] Hxt Jpeg Kbipack LinuxJoystick MathEx[contrib] Mesh Metis[contrib] Mmg Mpeg Netgen Nii2mesh ONELAB ONELABMetamodel OpenCASCADE OpenCASCADE-CAF OpenGL OpenMP OptHom Parser Plugins Png Post QuadMeshingTools QuadTri Solver TetGen/BR TinyXML2[contrib] Untangle Voro++[contrib] WinslowUntangler Zlib tinyobjloader
FLTK version  : 1.3.8
OCC version   : 7.9.2
Packaged by   : root
Web site      : https://gmsh.info
Issue tracker : https://gitlab.onelab.info/gmsh/gmsh/issues
-------------------------------------------------------


In [41]:
mat = Material("plate", E=2e5, ν=0.3)
problem = Problem([mat], type=:PlaneStress)

Problem("furat", :PlaneStress, 2, 2, Material[Material("plate", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 122360, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :f)

In [42]:
supp1 = displacementConstraint("bottom", ux=0, uy=0)
supp2 = displacementConstraint("top", uy=0)
load0 = load("top", fx=1)
loadL = load("left", fy=-1)
loadR = load("right", fy=1)

LoadCondition("right", nothing, Dict{Symbol, Union{Nothing, Function, Number, ScalarField}}(:fy => 1, :fz => nothing, :fx => nothing))

In [43]:
q = solveDisplacement(problem, load=[load0, loadL, loadR], support=[supp1, supp2])

nodal VectorField
[0.000654499487328294; 6.608356190511673e-5; … ; 0.0007676782627192666; -3.577026752217057e-5;;]

In [44]:
S = solveStress(q)

elementwise TensorField
[[0.004664868927804061; 1.0024436179509275; … ; 0.0; 0.0;;], [0.004499044898171296; 1.0028615891791373; … ; 0.0; 0.0;;], [0.00045174094724537356; 1.0014739180338774; … ; 0.0; 0.0;;], [-0.17079879395353187; 1.13100305218969; … ; 0.0; 0.0;;], [-0.0006310762762467162; 1.000145349943306; … ; 0.0; 0.0;;], [0.025035532640922446; 0.998080916678272; … ; 0.0; 0.0;;], [0.00040006406985232775; 1.0007218121486192; … ; 0.0; 0.0;;], [-0.010538017738087402; 1.001871543224702; … ; 0.0; 0.0;;], [0.0005780013488393049; 1.0002961889595476; … ; 0.0; 0.0;;], [0.0012354751946964959; 1.0033442554791152; … ; 0.0; 0.0;;]  …  [-0.7221660405089853; 1.077492776570938; … ; 0.0; 0.0;;], [-0.24627101246464878; 0.5561383250906234; … ; 0.0; 0.0;;], [-1.7784831989651304; 1.6862316392261836; … ; 0.0; 0.0;;], [0.014373479944533526; 0.2910755981160924; … ; 0.0; 0.0;;], [-0.18258107915431854; 0.52901791433083; … ; 0.0; 0.0;;], [-0.7703128873821826; 1.1718478185694172; … ; 0.0; 0.0;;], [0.91043177429

In [45]:
S1 = elementsToNodes(S)

nodal TensorField
[5.128352051342582e-9; 6.891296642183531e-8; … ; 0.0; 0.0;;]

In [46]:
a = 5
T = 1
σR(x, y, z) = T * (1 - 4 * a^2 / (x^2 + y^2) + 3 * a^4 / (x^2 + y^2)^2) * sin(2 * atan(y, x))
σφ(x, y, z) = T * (-1 - 3 * a^4 / (x^2 + y^2)^2) * sin(2 * atan(y / x))
τRφ(x, y, z) = T * (1 + 2 * a^2 / (x^2 + y^2) - 3 * a^4 / (x^2 + y^2)^2) * cos(2 * atan(y, x))

τRφ (generic function with 1 method)

In [47]:
sR = field("plate", f=σR)
sφ = field("plate", f=σφ)
sRφ = field("plate", f=τRφ)

sR1 = scalarField(problem, [sR])
sφ1 = scalarField(problem, [sφ])
sRφ1 = scalarField(problem, [sRφ])

nodal ScalarField
[0.0; 0.9803920895932353; … ; -0.22818981395957574; -0.22465635338992232;;]

In [48]:
showStressResults(S, :s, visible=true)

showDoFResults(sR1, name="σR")
showDoFResults(sφ1, name="σφ")
showDoFResults(sRφ1, name="τRφ")

3

In [49]:
nx(x, y) = x
ny(x, y) = y
cs = CoordinateSystem([nx, ny])
Q = rotateNodes(problem, "plate", cs)

LowLevelFEM.Transformation(sparse([1, 2, 3, 4, 3, 4, 5, 6, 5, 6  …  244715, 244716, 244717, 244718, 244717, 244718, 244719, 244720, 244719, 244720], [1, 2, 3, 3, 4, 4, 5, 5, 6, 6  …  244716, 244716, 244717, 244717, 244718, 244718, 244719, 244719, 244720, 244720], [1.0, 1.0, -0.9950371902099892, -0.09950371902099892, 0.09950371902099892, -0.9950371902099892, 0.9950371902099892, -0.09950371902099892, 0.09950371902099892, 0.9950371902099892  …  -0.8550115733341447, -0.5186089176486175, -0.5111781465702526, 0.8594747829162885, -0.8594747829162885, -0.5111781465702526, -0.5173178695987247, 0.8557933289023915, -0.8557933289023915, -0.5173178695987247], 244720, 244720), 122360, 2)

In [50]:
S2 = Q' * S1 * Q;

In [51]:
e = fieldError(S)

nodal TensorField
[4.93910311629743e-10; 3.134300590042049e-10; … ; 0.0; 0.0;;]

In [52]:
showDoFResults(S2, :sx, name="σR")
showDoFResults(S2, :sy, name="σφ")
showDoFResults(S2, :sxy, name="τRφ")
showDoFResults(e, :scalar, name="Error")

7

In [53]:
openPostProcessor()

-------------------------------------------------------
Version       : 4.15.0-git
License       : GNU General Public License
Build OS      : Linux64-sdk
Build date    : 19700101
Build host    : amdci7.julia.csail.mit.edu
Build options : 64Bit ALGLIB[contrib] ANN[contrib] Bamg Blossom Cairo DIntegration Dlopen DomHex Eigen[contrib] Fltk GMP Gmm[contrib] Hxt Jpeg Kbipack LinuxJoystick MathEx[contrib] Mesh Metis[contrib] Mmg Mpeg Netgen Nii2mesh ONELAB ONELABMetamodel OpenCASCADE OpenCASCADE-CAF OpenGL OpenMP OptHom Parser Plugins Png Post QuadMeshingTools QuadTri Solver TetGen/BR TinyXML2[contrib] Untangle Voro++[contrib] WinslowUntangler Zlib tinyobjloader
FLTK version  : 1.3.8
OCC version   : 7.9.2
Packaged by   : root
Web site      : https://gmsh.info
Issue tracker : https://gitlab.onelab.info/gmsh/gmsh/issues
-------------------------------------------------------
